# Лекција 11 - Протокол контекста модела (MCP)

**Протокол контекста модела (MCP)** је отворени стандард који омогућава агентима да динамички откривају и користе алате, ресурсе и изворе података у време извршавања. Уместо да се алати тврдокодирају у агенту, MCP омогућава агентима да се повежу на спољне сервере који на захтев излажу могућности.

У овој лекцији ћете научити:
- Шта је MCP и зашто је важан за агентске системе
- Како функционише MCP клијент-сервер архитектура
- Како изградити агенте који користе откривање алата у стилу MCP-а


## Подешавање

**Предуслови:**
- Azure AI Foundry пројекат са распоређеним моделом
- Покрените `az login` за аутентификацију


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated
from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Шта је Model Context Protocol (MCP)?

MCP дефинише стандардни начин да агенти вештачке интелигенције открију и комуницирају са спољним алатима и изворима података:

- **MCP Server**: Излаже алате, ресурсе и упите преко стандардног протокола
- **MCP Client**: Извршно окружење агента које се повезује на сервере и открива расположиве могућности
- **Dynamic Discovery**: Агенти не морају да имају уграђене алате — они откривају шта је доступно током извршавања

Ово је снажно за изградњу проширивих система агената где се нове могућности могу додати без измене кода агента.


## Како MCP функционише

```
┌─────────────┐     discover      ┌─────────────────┐
│  MCP Client  │ ──────────────► │   MCP Server     │
│  (Agent)     │                  │  (Tool Provider) │
│              │ ◄────────────── │                   │
│              │   tool results   │  • list_tools()  │
│              │                  │  • call_tool()   │
└─────────────┘                  │  • resources     │
                                  └─────────────────┘
```

1. Агенс (MCP клијент) се повезује са MCP сервером
2. Сервер одговара листом доступних алата и њихових шема
3. Агенс затим може да позове било који откривени алат током свог расуђивања
4. Резултати се враћају кроз исти протокол


## Симулирање откривања алата MCP

Пошто прави MCP сервер захтева покренут серверски процес, показаћемо образац користећи функције `@tool` које симулирају оно што би MCP-повезана услуга за смештај обезбедила.

У продукцији, ови алати би се динамички откривали са MCP сервера уместо да буду дефинисани локално.


In [ ]:
@tool(approval_mode="never_require")
def search_accommodations(
    location: Annotated[str, "The city to search for accommodations"],
    check_in: Annotated[str, "Check-in date (YYYY-MM-DD)"],
    check_out: Annotated[str, "Check-out date (YYYY-MM-DD)"],
    guests: Annotated[int, "Number of guests"] = 2
) -> str:
    """Search for accommodations (simulating an MCP-connected Airbnb tool).
    In production, this would be discovered via MCP from an accommodation service."""
    listings = {
        "Tokyo": [
            {"name": "Shinjuku Modern Apartment", "price": 120, "rating": 4.8},
            {"name": "Traditional Ryokan in Asakusa", "price": 200, "rating": 4.9},
            {"name": "Shibuya Studio", "price": 85, "rating": 4.5},
        ],
        "Paris": [
            {"name": "Le Marais Charming Flat", "price": 150, "rating": 4.7},
            {"name": "Montmartre Artist Loft", "price": 110, "rating": 4.6},
        ],
        "Barcelona": [
            {"name": "Gothic Quarter Penthouse", "price": 130, "rating": 4.8},
            {"name": "Barceloneta Beach Flat", "price": 95, "rating": 4.4},
        ],
    }
    results = listings.get(location, [])
    if not results:
        return f"No accommodations found in {location}"
    output = f"Accommodations in {location} ({check_in} to {check_out}, {guests} guests):\n"
    for listing in results:
        output += f"  - {listing['name']}: ${listing['price']}/night (★{listing['rating']})\n"
    return output


@tool(approval_mode="never_require")
def get_local_experiences(
    location: Annotated[str, "The city to find experiences in"],
    interest: Annotated[str, "Type of experience (food, culture, adventure, etc.)"] = "all"
) -> str:
    """Get local experiences and activities (simulating an MCP-connected tourism tool)."""
    experiences = {
        "Tokyo": {
            "food": ["Tsukiji Market Tour ($45)", "Ramen Making Class ($60)", "Sake Tasting ($35)"],
            "culture": ["Tea Ceremony ($50)", "Samurai Museum ($15)", "Sumo Tournament ($80)"],
            "adventure": ["Mt. Fuji Day Trip ($120)", "Go-kart City Tour ($80)"],
        },
        "Paris": {
            "food": ["Wine & Cheese Tasting ($55)", "Cooking Class ($90)", "Market Tour ($40)"],
            "culture": ["Louvre Guided Tour ($35)", "Montmartre Art Walk ($25)"],
        },
    }
    city_exp = experiences.get(location, {})
    if not city_exp:
        return f"No experiences found in {location}"
    if interest != "all" and interest in city_exp:
        items = city_exp[interest]
        return f"{interest.title()} experiences in {location}:\n" + "\n".join(f"  - {e}" for e in items)
    output = f"All experiences in {location}:\n"
    for cat, items in city_exp.items():
        output += f"\n  {cat.title()}:\n"
        for item in items:
            output += f"    - {item}\n"
    return output

## Изградња агента са алаткама у стилу MCP


In [ ]:
agent = await provider.create_agent(
    tools=[search_accommodations, get_local_experiences],
    name="AccommodationAgent",
    instructions="""You are an accommodation and travel experiences specialist powered by MCP-connected services.

Help travelers find the perfect place to stay and things to do. When searching:
1. Use the search_accommodations tool to find listings
2. Use the get_local_experiences tool to suggest activities
3. Compare options and make personalized recommendations
4. Consider the traveler's budget, interests, and travel style""",
)

response = await agent.run(
    "I'm visiting Tokyo for 5 nights in April with my partner. We love traditional Japanese culture and food. "
    "Find us a place to stay and suggest some experiences.",
    )
print(response)

## MCP у продукцији

У продукционом окружењу, MCP омогућава моћне шаблоне:

- **Динамично откривање алата**: Агенти се повезују са MCP серверима и откривају алате у време извршавања
- **Одвојена архитектура**: Провајдери алата могу да ажурирају независно од агента
- **Заједничко коришћење међу организацијама**: Тимови могу да изложе могућности преко MCP сервера које било који агент може да користи
- **Подршка Microsoft Agent Framework-а**: MAF има уграђену подршку за MCP клијента преко `mcp` интеграције

Да бисте користили стварни MCP сервер са MAF-ом, повезали бисте се преко `hosted_mcp_tool()` или MCP клијент интеграције.

**Сазнајте више:**
- [Спецификација MCP-а](https://modelcontextprotocol.io/)
- [Подршка за MCP у Microsoft Agent Framework-у](https://github.com/microsoft/agent-framework/tree/main/python/samples/02-agents/mcp)


## Резиме

У овој лекцији сте научили:
- **MCP** је отворени стандард за динамичко откривање алата између агената и пружалаца алата
- **клијент-сервер архитектура** омогућава агентима да током извршавања открију могућности
- MCP омогућава **прошириви, раздвојени агентски системи** где се алати могу додати без измена кода
- Microsoft Agent Framework обезбеђује **уграђену MCP подршку** за производну употребу


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Изјава о одрицању одговорности**:
Овај документ је преведен помоћу AI сервиса за превођење Co-op Translator (https://github.com/Azure/co-op-translator). Иако настојимо да преводи буду тачни, имајте у виду да аутоматски преводи могу да садрже грешке или нетачности. Оригинални документ на његовом изворном језику треба сматрати званичним извором. За критичне информације препоручује се професионални превод који изводи стручни преводилац. Не сносимо одговорност за било какве неспоразуме или погрешна тумачења која произлазе из коришћења овог превода.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
